## Setup & Dataset

In [5]:
import pandas as pd
import numpy as np
import time

# Sample based on your test data
data = {
    "name": [
        "وسام", "محمد", "ريم", "عبير", "براء", "يوسف", "بديعة", "رنتيس",
        "أحمد", "فاطمة", "علي", "سارة", "نور", "مجد", "عمر", "ليلى",
        "إحسان", "مريم", "نضال", "محمود", "ياسمين"
    ],
    "true_label": [
        "ذكر", "ذكر", "أنثى", "أنثى", "غير معروف", "ذكر", "أنثى", "غير معروف",
        "ذكر", "أنثى", "ذكر", "أنثى", "غير معروف", "غير معروف", "ذكر", "أنثى",
        "غير معروف", "أنثى", "ذكر", "ذكر", "أنثى"
    ]
}
df = pd.DataFrame(data)
df

,name,true_label
0,وسام,ذكر
1,محمد,ذكر
2,ريم,أنثى
3,عبير,أنثى
4,براء,غير معروف
5,يوسف,ذكر
6,بديعة,أنثى
7,رنتيس,غير معروف
8,أحمد,ذكر
9,فاطمة,أنثى


## Approach 1 - Improved LLM (Few-Shot Prompting)
One-shot prompting provides a single example to guide an AI, ideal for simple formatting or tone guidance. Few-shot prompting provides multiple (2–5+) examples, offering higher accuracy, better, and more consistent, results for complex, nuanced, or highly structured tasks. Both improve performance over zero-shot by minimizing ambiguity.

In [8]:
from llama_cpp import Llama
import json
import re

# Load model (assuming it's already downloaded via your script)
llm = Llama.from_pretrained(
    repo_id="codegood/gemma-2b-it-Q4_K_M-GGUF",
	filename="gemma-2b-it.Q4_K_M.gguf",
    n_ctx=2048,
    verbose=False
)

def predict_llm_few_shot(names_list):
    names_str = json.dumps(names_list, ensure_ascii=False)
    
    # Few-shot prompt forces the model to understand the pattern and Arabic output
    prompt = (
        "<start_of_turn>user\n"
        "Classify the gender of the following Arabic first names.\n"
        "Output ONLY a valid JSON dictionary mapping the name to 'ذكر', 'أنثى', or 'غير معروف'.\n\n"
        "Example Input:[\"أحمد\", \"فاطمة\", \"مجد\", \"نابلس\"]\n"
        "Example Output: {\"أحمد\": \"ذكر\", \"فاطمة\": \"أنثى\", \"مجد\": \"غير معروف\", \"نابلس\": \"غير معروف\"}\n\n"
        f"Input: {names_str}<end_of_turn>\n"
        "<start_of_turn>model\n{"
    )
    
    start_time = time.time()
    response = llm(prompt, max_tokens=512, stop=["<end_of_turn>"], temperature=0.0)
    latency = time.time() - start_time
    
    text_output = "{" + response['choices'][0]['text'].strip()
    json_match = re.search(r'\{.*?\}', text_output, re.DOTALL)
    
    try:
        return json.loads(json_match.group(0)), latency
    except Exception:
        return {}, latency

# Test it
llm_preds, llm_time = predict_llm_few_shot(df['name'].tolist())
df['llm_pred'] = df['name'].map(lambda x: llm_preds.get(x, "غير معروف"))
print(f"LLM Processing Time: {llm_time:.2f}s")

c:\Users\RasheedAlqubbaj\OneDrive - TAMER Institute for Community Education\Desktop\name_duplicate\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\RasheedAlqubbaj\OneDrive - TAMER Institute for Community Education\Desktop\name_duplicate\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_context: n_ctx_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


LLM Processing Time: 13.67s


In [12]:
# Evaluate accuracy
accuracy = (df['true_label'] == df['llm_pred']).mean()
print(f"LLM Accuracy: {accuracy:.2%}")

# Display some results
df.head(10)

LLM Accuracy: 57.14%


,name,true_label,llm_pred
0,وسام,ذكر,أنثى
1,محمد,ذكر,ذكر
2,ريم,أنثى,غير معروف
3,عبير,أنثى,أنثى
4,براء,غير معروف,ذكر
5,يوسف,ذكر,أنثى
6,بديعة,أنثى,أنثى
7,رنتيس,غير معروف,ذكر
8,أحمد,ذكر,ذكر
9,فاطمة,أنثى,أنثى


## Approach 2 - Heuristics + Dictionary (The Fastest Method)

Arabic names have strong morphological rules (e.g., Ta' Marbuta). We can catch 80% of names instantly without ML.

In [13]:
# A static dictionary for common names speeds up processing massively
MALE_DICT = {"محمد", "أحمد", "يوسف", "محمود", "علي", "عمر"}
FEMALE_DICT = {"ريم", "عبير", "مريم", "سارة", "نور"}
# Explicitly neutral or last names/cities
NEUTRAL_DICT = {"براء", "وسام", "نور", "رنتيس"}

def predict_heuristic(name):
    name = name.strip()
    
    # 1. Dictionary Lookup (O(1) time complexity)
    if name in NEUTRAL_DICT: return "غير معروف"
    if name in MALE_DICT: return "ذكر"
    if name in FEMALE_DICT: return "أنثى"
    
    # 2. Morphological Rules
    if name.startswith("عبد "): return "ذكر"
    if name.startswith("أبو "): return "ذكر"
    if name.startswith("أم "): return "أنثى"
    
    # Ta' Marbuta rule (most are female, exceptions exist like حمزة, أسامة)
    exceptions_ta = {"حمزة", "أسامة", "معاوية", "طلحة", "حذيفة", "عبيدة"}
    if name.endswith("ة"):
        return "ذكر" if name in exceptions_ta else "أنثى"
        
    return "غير معروف"

start_time = time.time()
df['heuristic_pred'] = df['name'].apply(predict_heuristic)
print(f"Heuristic Processing Time: {time.time() - start_time:.4f}s")

Heuristic Processing Time: 0.0022s


In [16]:
# Evaluate heuristic accuracy
heuristic_accuracy = (df['true_label'] == df['heuristic_pred']).mean()
print(f"Heuristic Accuracy: {heuristic_accuracy:.2%}")

# Display some results
df[['name', 'true_label', 'heuristic_pred']].head(10)

Heuristic Accuracy: 80.95%


,name,true_label,heuristic_pred
0,وسام,ذكر,غير معروف
1,محمد,ذكر,ذكر
2,ريم,أنثى,أنثى
3,عبير,أنثى,أنثى
4,براء,غير معروف,غير معروف
5,يوسف,ذكر,ذكر
6,بديعة,أنثى,أنثى
7,رنتيس,غير معروف,غير معروف
8,أحمد,ذكر,ذكر
9,فاطمة,أنثى,أنثى


Results are promising but nothing wowing me + I'd have to make the Dictionary

## Approach 3 - Character N-Gram ML
This approach learns the sub-word patterns of Arabic names. Train this once on a dataset of ~10,000 names

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# Example training data (In production, load a CSV of ~5000 names)
train_names =["محمد", "يوسف", "أحمد", "محمود", "علي", "فاطمة", "ريم", "عبير", "خديجة", "سعاد"]
train_labels =["ذكر", "ذكر", "ذكر", "ذكر", "ذكر", "أنثى", "أنثى", "أنثى", "أنثى", "أنثى"]

# analyzer='char_wb' extracts n-grams of characters only from inside words
# This catches prefixes and suffixes natively
model_ml = make_pipeline(
    CountVectorizer(analyzer='char_wb', ngram_range=(2, 4)),
    MultinomialNB()
)

model_ml.fit(train_names, train_labels)

# Test it
start_time = time.time()
df['ml_pred'] = model_ml.predict(df['name'])

# Overwrite with "غير معروف" if confidence is low (probability < 0.7)
probs = model_ml.predict_proba(df['name'])
max_probs = np.max(probs, axis=1)
df['ml_pred'] = np.where(max_probs < 0.65, "غير معروف", df['ml_pred'])

print(f"ML Processing Time: {time.time() - start_time:.4f}s")

ML Processing Time: 0.0086s
